# EDA: Wongnai restaurant reviews

Runs on the processed splits written by `python -m src.data`.

Star distribution and review length describe the data source, so they are computed on the
whole dataset. Word lists and example reviews use the train split only, so nothing from
val/test leaks into modeling decisions.

Figures are written to `results/figures/` and `report/figures/` as `eda_*.png`.

In [ ]:
import re
import sys
import time
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import config, data, plotting

plotting.setup()
pd.set_option("display.max_colwidth", 200)

## 1. Splits

Check that the stratified split kept the class shares the same everywhere.

In [ ]:
splits = {}
for name in ("train", "val", "test"):
    splits[name] = data.load_split(name)

rows = []
for name, part in splits.items():
    counts = part["label"].value_counts()
    row = {"split": name, "rows": len(part)}
    for label in config.LABELS:
        row[label] = f"{counts[label]:,} ({counts[label] / len(part):.1%})"
    rows.append(row)

split_table = pd.DataFrame(rows).set_index("split")
print(f"total rows: {sum(len(p) for p in splits.values()):,}")
split_table

In [ ]:
full = pd.concat(splits.values(), ignore_index=True)
train = splits["train"]
print(f"full {len(full):,} reviews, train {len(train):,} reviews")

## 2. Star and class distribution

In [ ]:
star_counts = full["star"].value_counts().sort_index()
label_counts = full["label"].value_counts().reindex(config.LABELS)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(star_counts.index, star_counts.values, color=config.BAR_COLOR)
axes[0].set_xlabel("ดาว")
axes[0].set_ylabel("จำนวนรีวิว")
axes[0].set_title("การกระจายของดาว")
for star, count in star_counts.items():
    text = f"{count:,}\n{count / len(full):.1%}"
    axes[0].text(star, count, text, ha="center", va="bottom", fontsize=8)

label_colors = [config.LABEL_COLORS[label] for label in config.LABELS]
axes[1].bar(config.LABELS, label_counts.values, color=label_colors)
axes[1].set_xlabel("คลาส")
axes[1].set_title("การกระจายของ 3 คลาส")
for i, count in enumerate(label_counts.values):
    text = f"{count:,}\n{count / len(full):.1%}"
    axes[1].text(i, count, text, ha="center", va="bottom", fontsize=8)

for ax in axes:
    ax.margins(y=0.18)
fig.tight_layout()
plotting.save(fig, "eda_class_distribution")

print(pd.DataFrame({"count": star_counts, "share": star_counts / len(full)}))
print()
print(pd.DataFrame({"count": label_counts, "share": label_counts / len(full)}))

The majority class alone is ~64% of the data, so accuracy is a weak metric here.
A model that always answers `positive` would already score that. Macro-F1 is the
main metric because it gives the 5.6% negative class the same weight as the rest.

## 3. Review length in characters

In [ ]:
full["n_chars"] = full["text"].str.len()
full["n_chars"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0)

In [ ]:
CHAR_AXIS_LIMIT = 3000

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(full["n_chars"], bins=80, range=(0, CHAR_AXIS_LIMIT), color=config.BAR_COLOR)
ax.set_xlabel("ความยาวรีวิว (ตัวอักษร)")
ax.set_ylabel("จำนวนรีวิว")
ax.set_title(
    f"ความยาวรีวิว - median {full['n_chars'].median():,.0f} ตัวอักษร"
    f" (ตัดแกนที่ {CHAR_AXIS_LIMIT:,})"
)
fig.tight_layout()
plotting.save(fig, "eda_length_chars")

## 4. Review length in bge-m3 tokens

This decides `EMBEDDING_MAX_LENGTH`. Anything above it gets cut off before the
embedding model sees it, so the share of reviews over the limit has to be reported.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.EMBEDDING_MODEL)

texts = full["text"].tolist()
token_counts = []
started = time.perf_counter()
for start in range(0, len(texts), 1000):
    batch = texts[start:start + 1000]
    encoded = tokenizer(batch, add_special_tokens=True, truncation=False)
    for ids in encoded["input_ids"]:
        token_counts.append(len(ids))

full["n_tokens"] = token_counts
print(f"tokenized {len(texts):,} reviews in {time.perf_counter() - started:.1f}s")
full["n_tokens"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0)

In [ ]:
over_limit = full["n_tokens"] > config.EMBEDDING_MAX_LENGTH
print(f"max_length = {config.EMBEDDING_MAX_LENGTH}")
print(f"reviews over the limit: {over_limit.sum():,} ({over_limit.mean():.2%})")
print(f"tokens per character: {full['n_tokens'].sum() / full['n_chars'].sum():.3f}")
print()
print("share over the limit per class")
for label in config.LABELS:
    part = full[full["label"] == label]
    share = (part["n_tokens"] > config.EMBEDDING_MAX_LENGTH).mean()
    print(f"  {label:<9} {share:.2%}")

In [ ]:
TOKEN_AXIS_LIMIT = 1500

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(full["n_tokens"], bins=80, range=(0, TOKEN_AXIS_LIMIT), color=config.BAR_COLOR)
ax.axvline(config.EMBEDDING_MAX_LENGTH, color="#c2255c", linestyle="--")
ax.annotate(
    f"max_length = {config.EMBEDDING_MAX_LENGTH}\nถูกตัด {over_limit.mean():.1%} ของรีวิว",
    xy=(config.EMBEDDING_MAX_LENGTH, 0),
    xytext=(config.EMBEDDING_MAX_LENGTH + 60, ax.get_ylim()[1] * 0.75),
    color="#c2255c",
)
ax.set_xlabel("ความยาวรีวิว (token ของ bge-m3)")
ax.set_ylabel("จำนวนรีวิว")
ax.set_title(f"ความยาวรีวิวเป็น token (ตัดแกนที่ {TOKEN_AXIS_LIMIT:,})")
fig.tight_layout()
plotting.save(fig, "eda_length_tokens")

## 5. Length per class

If one class is systematically longer, length itself carries signal and the model
may pick up on it instead of on sentiment.

In [ ]:
token_groups = []
for label in config.LABELS:
    token_groups.append(full.loc[full["label"] == label, "n_tokens"].to_numpy())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

box = axes[0].boxplot(token_groups, showfliers=False, patch_artist=True)
for patch, label in zip(box["boxes"], config.LABELS):
    patch.set_facecolor(config.LABEL_COLORS[label])
for median in box["medians"]:
    median.set_color("black")
axes[0].set_xticks(range(1, len(config.LABELS) + 1))
axes[0].set_xticklabels(config.LABELS)
axes[0].set_ylabel("จำนวน token")
axes[0].set_title("การกระจายความยาวแยกตามคลาส (ไม่แสดง outlier)")

medians = []
for group in token_groups:
    medians.append(float(pd.Series(group).median()))
axes[1].bar(config.LABELS, medians, color=label_colors)
axes[1].set_ylabel("median token")
axes[1].set_title("ความยาวกลางแยกตามคลาส")
for i, value in enumerate(medians):
    axes[1].text(i, value, f"{value:,.0f}", ha="center", va="bottom", fontsize=9)
axes[1].margins(y=0.15)

fig.tight_layout()
plotting.save(fig, "eda_length_by_class")

summary = full.groupby("label")[["n_chars", "n_tokens"]].median()
summary.reindex(config.LABELS).round(0)

## 6. Frequent words per class (train only)

Tokenized with `newmm`, Thai stopwords removed. Raw counts are dominated by generic
food words, so a distinctiveness score (share inside the class vs share outside it)
is computed as well.

In [ ]:
from pythainlp.corpus import thai_stopwords
from pythainlp.tokenize import word_tokenize

stopwords = set(thai_stopwords())
# Drop punctuation, emoji and single characters; keep Thai, Latin and digits
WORD_PATTERN = re.compile(r"^[\u0E00-\u0E7Fa-zA-Z0-9]{2,}$")

counters = {}
for label in config.LABELS:
    counters[label] = Counter()

started = time.perf_counter()
for text, label in zip(train["text"], train["label"]):
    for token in word_tokenize(text, engine="newmm", keep_whitespace=False):
        token = token.strip().lower()
        if token in stopwords or not WORD_PATTERN.match(token):
            continue
        counters[label][token] += 1

print(f"tokenized {len(train):,} train reviews in {time.perf_counter() - started:.1f}s")
for label in config.LABELS:
    print(f"  {label:<9} {sum(counters[label].values()):>9,} tokens kept")

In [ ]:
TOP_N = 15

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, label in zip(axes, config.LABELS):
    top = counters[label].most_common(TOP_N)
    words = [word for word, _ in top][::-1]
    counts = [count for _, count in top][::-1]
    ax.barh(words, counts, color=config.LABEL_COLORS[label])
    ax.set_title(label)
    ax.set_xlabel("จำนวนครั้ง")

fig.suptitle(f"คำที่พบบ่อยที่สุด {TOP_N} คำในแต่ละคลาส (train)")
fig.tight_layout()
plotting.save(fig, "eda_top_words")

In [ ]:
MIN_COUNT = 50

overall = Counter()
for label in config.LABELS:
    overall.update(counters[label])
total_tokens = sum(overall.values())

distinctive = {}
for label in config.LABELS:
    inside_total = sum(counters[label].values())
    outside_total = total_tokens - inside_total
    scored = []
    for word, overall_count in overall.items():
        if overall_count < MIN_COUNT:
            continue
        inside = counters[label][word]
        outside = overall_count - inside
        # +1 smoothing so words absent outside the class do not blow up
        ratio = ((inside + 1) / inside_total) / ((outside + 1) / outside_total)
        scored.append((ratio, word, inside))
    scored.sort(reverse=True)
    distinctive[label] = scored[:TOP_N]

for label in config.LABELS:
    print(f"===== {label} =====")
    for ratio, word, inside in distinctive[label]:
        print(f"  {word:<20} x{ratio:>5.2f}  ({inside:,} ครั้ง)")
    print()

## 7. Example reviews (train only)

In [ ]:
PREVIEW_CHARS = 350

for label in config.LABELS:
    part = train[train["label"] == label]
    print(f"===== {label} ({len(part):,} reviews) =====")
    for _, row in part.sample(3, random_state=config.SEED).iterrows():
        print(f"[{row['star']} ดาว, {len(row['text']):,} ตัวอักษร]")
        print(row["text"][:PREVIEW_CHARS])
        print()

## 8. What this means for the next steps

Filled in after running the notebook.